In [1]:
!find /kaggle/input -name 'classes.csv' -o -name '*.zip'

/kaggle/input/datasets/bxigflor/pa3-dados/pa3_dados/classes.csv


In [3]:
%pip install -q open_clip_torch

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
import csv
import json
import time
from importlib.metadata import version

import matplotlib.pyplot as plt
import open_clip
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_DIR = Path("/kaggle/input/datasets/bxigflor/pa3-dados/pa3_dados")  # Cambia por la ruta real de tu dataset.
OUTPUT_DIR = Path('/kaggle/working/teachers_outputs')
MODEL_ID = 'hf-hub:imageomics/bioclip'
BATCH_SIZE = 32
NUM_WORKERS = 2  # Usa 0 si tu entorno tiene problemas al crear procesos.

classes_path = DATA_DIR / 'classes.csv'

with classes_path.open(newline='', encoding='utf-8') as file:
    class_rows = sorted(csv.DictReader(file), key=lambda row: int(row['idx']))
class_names = [row['label'] for row in class_rows]

image_paths = sorted((DATA_DIR / 'images' / 'train').glob('*.jpg'))
device = 'cuda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Fotos:', len(image_paths))

GPU: Tesla T4
Fotos: 6505


## Profesor y prompts

El notebook clona el repositorio en la carpeta de trabajo e importa `build_prompts`
desde `teacher_prompts.py`. El archivo debe estar en la rama indicada por `REPO_BRANCH`.
Si ya clonaste el repositorio en esta sesión, se reutiliza esa copia.
El orden de los textos coincide con los índices de `classes.csv`.
El primer uso descarga los pesos de BioCLIP.

In [14]:
import subprocess
import sys

import open_clip

REPO_URL = "https://github.com/nicolasspaniol/deep-learning-pa3.git"
REPO_DIR = OUTPUT_DIR.parent / "deep-learning-pa3"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "src"))

from deep_learning_pa3.teachers_prompts import build_prompts

prompts = build_prompts(class_names)
print(f"{len(prompts)} prompts. Ejemplo: {prompts[0]}")

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_ID, device=device)
model.eval()

tokenizer = open_clip.get_tokenizer(MODEL_ID)
with torch.inference_mode():
    text_features = model.encode_text(tokenizer(prompts).to(device), normalize=True)

Cloning into '/kaggle/working/deep-learning-pa3'...


21 prompts. Ejemplo: a camera trap photo of the american black bear.


open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

## Inferencia por lotes y exportación

Cada fila contiene `image_id`, `label`, `confidence` y una columna `prob_<clase>` por clase.
`confidence` es el máximo softmax entre las 21 opciones y expresa confianza relativa.
Conservamos todas las predicciones para probar filtros después, usando el desempeño del alumno en val.
Durante el entrenamiento del alumno, las etiquetas reales de su presupuesto tienen prioridad.

La barra de progreso muestra una estimación del tiempo restante. El CSV final se publica
cuando termina el recorrido completo. Si la sesión se interrumpe, vuelve a ejecutar esta celda.

In [15]:
from PIL import Image
from tqdm.auto import tqdm

csv_path = OUTPUT_DIR / "teacher_train.csv"
temporary_path = csv_path.with_suffix(".csv.tmp")

with temporary_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["image_id", "label", "confidence"])

    with torch.inference_mode():
        for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="BioCLIP en train"):
            batch_paths = image_paths[start:start + BATCH_SIZE]
            tensors = []
            for image_path in batch_paths:
                with Image.open(image_path) as image:
                    tensors.append(preprocess(image.convert("RGB")))

            images = torch.stack(tensors).to(device)
            image_features = model.encode_image(images, normalize=True)
            similarities = image_features @ text_features.T
            logits = model.logit_scale.exp() * similarities
            probabilities = logits.float().softmax(dim=1)
            confidence, prediction = probabilities.max(dim=1)

            for image_path, class_index, score in zip(
                batch_paths, prediction.tolist(), confidence.tolist(), strict=True
            ):
                writer.writerow([image_path.stem, class_names[class_index], score])

temporary_path.replace(csv_path)
print(f"Guardadas {len(image_paths)} pseudoetiquetas en {csv_path}")

BioCLIP en train:   0%|          | 0/204 [00:00<?, ?it/s]

Guardadas 6505 pseudoetiquetas en /kaggle/working/teachers_outputs/teacher_train.csv
